# Kabwe Municipal Council — IDP & Strategic Community Projects

**CSC 4792: Data Mining and Warehousing · 2025/26**
**Workstream:** Integrated Development Plans (IDPs) & Strategic Community Project Records

## Overview

In this notebook we extract, clean, and curate information about Kabwe Municipal
Council's Integrated Development Plan (2023–2033) and its strategic community
project records. Our pipeline has three phases:

**Phase 1 — Discovery** — We crawl the council website with BeautifulSoup to
identify which PDFs are published and where they live. This produces a URL list,
not the data itself.

**Phase 2 — Extraction** — We read the four PDFs (placed manually in
`data/raw/idp/`) and pull out structured tables covering IDP strategic areas,
development goals, baseline statistics, sub-programmes, and community project
registries.

**Phase 3 — Cleaning & Preprocessing** — We transform the raw extraction output
into an analysis-ready dataset: standardised, deduplicated, enriched, and
exported as pipe-delimited CSVs.

We keep the phases separate so each stage can be re-run independently.

## Phase 1 · Setup

We import the libraries needed for discovery and set up the folder structure.
Our notebook can run from either the repo root or from inside `notebooks/`, so
we detect the current directory and resolve paths relative to the repo root
accordingly.

Folders we create:

- `data/raw/idp/` — where the downloaded PDFs live
- `data/discovered/` — where we save the URL list
- `data/extracted/` — Phase 2's intermediate output
- `data/clean/` — Phase 3's intermediate output
- `outputs/` — the final CSVs for Kaggle

In [19]:
# ============================================================
# PHASE 1 · CELL 1 — SETUP
# ============================================================
# !pip install requests beautifulsoup4

import os
import re
import json
import warnings
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

try:
    from requests.exceptions import RequestsDependencyWarning
    warnings.filterwarnings("ignore", category=RequestsDependencyWarning)
except ImportError:
    pass


# ---------- Repo root (hardcoded) ----------
REPO_ROOT = Path(r"C:\Users\dell\Desktop\Group6\group6_administration")

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")


# ---------- Folder structure ----------
DIRS = {
    "raw":        REPO_ROOT / "data" / "raw" / "IDP",
    "discovered": REPO_ROOT / "data" / "discovered",
    "extracted":  REPO_ROOT / "data" / "extracted" / "IDP",
    "clean":      REPO_ROOT / "data" / "cleaned" / "IDP",
    "outputs":    REPO_ROOT / "data" / "processed" / "IDP",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

PROJECT_CODE = "db-unza26-csc4792"
COUNCIL_SLUG = "kabwe"
COUNCIL_FULL = "Kabwe Municipal Council"
CONSTITUENCY = "Kabwe Central"
BASE_URL     = "https://www.kabwecouncil.gov.zm"
RUN_STAMP    = datetime.now().isoformat(timespec="seconds")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

print("PHASE 1 — DISCOVERY")
print(f"  Repo root : {REPO_ROOT}")
print(f"  PDFs dir  : {DIRS['raw']}")
print(f"  Exists    : {DIRS['raw'].exists()}")

# Quick check: are the 4 PDFs there?
if DIRS["raw"].exists():
    pdfs = list(DIRS["raw"].glob("*.pdf"))
    print(f"  PDFs found: {len(pdfs)}")
    for p in pdfs:
        print(f"    • {p.name}  ({p.stat().st_size / 1024:.1f} KB)")
else:
    print(f"  Folder does not exist yet — will be created")

PHASE 2 — EXTRACTION
  Repo root : C:\Users\dell\Desktop\Group6\group6_administration
  PDFs dir  : C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP
  Out dir   : C:\Users\dell\Desktop\Group6\group6_administration\data\extracted\IDP
  Sources   : 4


## Phase 1 · Fetching the council homepage

We start by fetching the council's homepage HTML — our crawling entry point,
the page from which we discover what documents the council links to.

We've observed that the council's server presents an incomplete SSL certificate
chain, so we disable certificate verification. This is safe because we only read
public documents — no credentials, no sensitive data.

In [12]:
# ============================================================
# PHASE 1 · CELL 2 — FETCH HOMEPAGE
# ============================================================

def fetch_html(url, verify_ssl=False, timeout=30):
    try:
        r = requests.get(
            url, headers=HEADERS, timeout=timeout,
            verify=verify_ssl, allow_redirects=True,
        )
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  [error] {url}  →  {type(e).__name__}: {e}")
        return None


homepage_html = fetch_html(BASE_URL)

if homepage_html:
    soup  = BeautifulSoup(homepage_html, "html.parser")
    title = soup.find("title")
    print(f"Homepage fetched: {len(homepage_html):,} chars")
    print(f"Title: {title.string.strip() if title else '(no title)'}")
else:
    print("Homepage fetch failed.")

C:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Homepage fetched: 167,369 chars
Title: Kabwe Municipal Council – Kabwe


## Phase 1 · Discovering PDF links

We parse the homepage's HTML for anchor tags pointing to PDF files. We convert
relative URLs to absolute, then filter out links to other domains — a common
issue when council pages link to news about other councils.

This is the classic HTML-scraping workflow: fetch, parse, filter.

In [20]:
# ============================================================
# PHASE 1 · CELL 3 — DISCOVER PDF LINKS
# ============================================================

def discover_pdf_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    base_domain = urlparse(base_url).netloc
    found = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href.lower().endswith(".pdf"):
            continue
        absolute = urljoin(base_url, href)
        if urlparse(absolute).netloc != base_domain:
            continue
        found.add(absolute)

    return sorted(found)


homepage_pdfs = discover_pdf_links(homepage_html, BASE_URL) if homepage_html else []

print(f"PDF links discovered on homepage: {len(homepage_pdfs)}")
for u in homepage_pdfs:
    print(f"  • {u}")

PDF links discovered on homepage: 1
  • https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/09/CDF-Guidelines.pdf


## Phase 1 · Curated source list

The homepage links only two PDFs. When we checked manually, we found that the
IDP and project documents we need are hosted at direct URLs under
`/wp-content/uploads/` — but they aren't linked from any crawlable index page.

So we combine two sources of truth:

1. **Dynamic discovery** — what the crawl finds on the homepage
2. **Curated list** — the four documents we've verified manually

This is standard practice when scraping council and government sites: discovery
finds what's new, curation guarantees the specific documents we need.

In [21]:
# ============================================================
# PHASE 1 · CELL 4 — CURATED SOURCES
# ============================================================

CURATED_SOURCES = [
    {
        "doc_id":   "IDP_MAIN",
        "title":    "Kabwe Approved IDP Final Version 1 (2023–2033)",
        "category": "Integrated Development Plan",
        "filename": "Kabwe-Approved-IDP_Final-Version-1.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf",
    },
    {
        "doc_id":   "IDP_CITIZEN",
        "title":    "Kabwe District Citizen IDP",
        "category": "Integrated Development Plan",
        "filename": "Kabwe-District-Citizen-IDP_Final-1.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2024/09/Kabwe-District-Citizen-IDP_Final-1.pdf",
    },
    {
        "doc_id":   "CDF_PROJECTS_2025",
        "title":    "2025 Approved Community Projects — Kabwe Central",
        "category": "Strategic Community Projects",
        "filename": "2025-Approved-Community-Projects_Kabwe-Central.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2025/06/2025-Approved-Community-Projects_Kabwe-Central.pdf",
    },
    {
        "doc_id":   "NEWSLETTER_2024",
        "title":    "Kabwe Municipal Council Newsletter 2024",
        "category": "Council Newsletter",
        "filename": "Kabwe-Municipal-Council-Newsletter-2024-1.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2024/09/Kabwe-Municipal-Council-Newsletter-2024-1.pdf",
    },
]

print(f"Curated sources: {len(CURATED_SOURCES)}")
for s in CURATED_SOURCES:
    print(f"  [{s['doc_id']:<18}] {s['title']}")

Curated sources: 4
  [IDP_MAIN          ] Kabwe Approved IDP Final Version 1 (2023–2033)
  [IDP_CITIZEN       ] Kabwe District Citizen IDP
  [CDF_PROJECTS_2025 ] 2025 Approved Community Projects — Kabwe Central
  [NEWSLETTER_2024   ] Kabwe Municipal Council Newsletter 2024


## Phase 1 · Saving the discovery result

We write the combined URL list to `data/discovered/sources.json` so Phase 2 can
read it without re-crawling. This keeps the two phases decoupled — if we later
re-run Phase 2, we don't need network access.

In [22]:
# ============================================================
# PHASE 1 · CELL 5 — SAVE DISCOVERY RESULT
# ============================================================

payload = {
    "run_at":          RUN_STAMP,
    "base_url":        BASE_URL,
    "homepage_pdfs":   homepage_pdfs,
    "curated_sources": CURATED_SOURCES,
}

out_file = DIRS["discovered"] / "sources.json"
out_file.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(f"   Saved: {out_file.resolve()}")
print(f"   HTML-discovered : {len(homepage_pdfs)}")
print(f"   Curated         : {len(CURATED_SOURCES)}")

   Saved: C:\Users\dell\Desktop\Group6\group6_administration\data\discovered\sources.json
   HTML-discovered : 1
   Curated         : 4


## Phase 1 · Manual download instructions

We deliberately do **not** download the PDFs programmatically. In our earlier
attempts, the council's server rejected scripted downloads due to SSL issues and
bandwidth throttling. Instead, we download the files manually via our browser
and place them in `data/raw/idp/`.

The next cell prints the exact URLs and folder path so we can copy them into a
browser.

In [23]:
# ============================================================
# PHASE 1 · CELL 6 — MANUAL DOWNLOAD INSTRUCTIONS
# ============================================================

target = DIRS["raw"].resolve()

print("=" * 72)
print("MANUAL DOWNLOAD INSTRUCTIONS")
print("=" * 72)
print(f"\nTarget folder: {target}\n")
for s in CURATED_SOURCES:
    print(f"  • {s['filename']}")
    print(f"    {s['url']}\n")
print("=" * 72)

MANUAL DOWNLOAD INSTRUCTIONS

Target folder: C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP

  • Kabwe-Approved-IDP_Final-Version-1.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf

  • Kabwe-District-Citizen-IDP_Final-1.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Citizen-IDP_Final-1.pdf

  • 2025-Approved-Community-Projects_Kabwe-Central.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects_Kabwe-Central.pdf

  • Kabwe-Municipal-Council-Newsletter-2024-1.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Municipal-Council-Newsletter-2024-1.pdf



## Phase 1 · Verifying the PDFs are in place

Once we've downloaded the four PDFs, we run this cell to verify they're in the
right folder with the right filenames. If anything is missing, the cell tells us
exactly which file and where to put it.

Once this cell passes, Phase 1 is complete.

In [24]:
# ============================================================
# PHASE 1 · CELL 7 — VERIFY PDFs ON DISK
# ============================================================

REQUIRED_FILES = [s["filename"] for s in CURATED_SOURCES]

print(f"Checking: {DIRS['raw'].resolve()}\n")

present = sorted(f.name for f in DIRS["raw"].iterdir()
                 if f.is_file() and f.suffix.lower() == ".pdf")

print(f"PDFs found: {len(present)}")
for f in present:
    size_kb = (DIRS["raw"] / f).stat().st_size / 1024
    marker  = "✓" if f in REQUIRED_FILES else "?"
    print(f"  {marker} {f}  ({size_kb:.1f} KB)")

missing = [f for f in REQUIRED_FILES if f not in present]
if missing:
    print(f"\nMissing {len(missing)} file(s):")
    for f in missing:
        print(f"   - {f}")
    raise FileNotFoundError("Download missing PDFs before continuing.")
else:
    print(f"\nAll {len(REQUIRED_FILES)} PDFs present.")
    print("   Phase 1 complete. Proceed to Phase 2.")

Checking: C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP

PDFs found: 4
  ✓ 2025-Approved-Community-Projects_Kabwe-Central.pdf  (27.2 KB)
  ✓ Kabwe-Approved-IDP_Final-Version-1.pdf  (17202.6 KB)
  ✓ Kabwe-District-Citizen-IDP_Final-1.pdf  (41227.1 KB)
  ✓ Kabwe-Municipal-Council-Newsletter-2024-1.pdf  (2703.6 KB)

All 4 PDFs present.
   Phase 1 complete. Proceed to Phase 2.


## Phase 2 · Section 2.0 — Setup

We import `pdfplumber` and `pandas`, resolve the folder paths the same way as
Phase 1, and reload the curated source list from the discovery JSON.

We then read every page of every PDF into memory. For each page we keep both
the page number and the page text. The page number lets us trace any extracted
fact back to its source.

**Note:** Phase 2 does not touch the network. Everything reads from disk.

In [28]:
# ============================================================
# PHASE 2 · CELL 1 — SETUP
# ============================================================
import os
import re
import json
import warnings
from pathlib import Path

import pandas as pd
import pdfplumber

warnings.filterwarnings("ignore", category=Warning)

# ---------- Repo root (hardcoded) ----------
REPO_ROOT = Path(r"C:\Users\dell\Desktop\Group6\group6_administration")

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

# ---------- Paths ----------
RAW_DIR       = REPO_ROOT / "data" / "raw" / "IDP"
EXTRACTED_DIR = REPO_ROOT / "data" / "extracted" / "IDP"
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Reload curated sources ----------
sources_payload = json.loads(
    (REPO_ROOT / "data" / "discovered" / "sources.json").read_text()
)
CURATED_SOURCES = sources_payload["curated_sources"]
sources_by_id   = {s["doc_id"]: s for s in CURATED_SOURCES}

print("PHASE 2 — EXTRACTION")
print(f"  Repo root : {REPO_ROOT}")
print(f"  PDFs dir  : {RAW_DIR}")
print(f"  Out dir   : {EXTRACTED_DIR}")
print(f"  Sources   : {len(CURATED_SOURCES)}")

PHASE 2 — EXTRACTION
  Repo root : C:\Users\dell\Desktop\Group6\group6_administration
  PDFs dir  : C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP
  Out dir   : C:\Users\dell\Desktop\Group6\group6_administration\data\extracted\IDP
  Sources   : 4


## Phase 2 · Section 2.1 — Extraction Utilities

We define the helpers every extractor below will use:

- **`make_unique_columns()`** — PDF tables often have empty or duplicate column
  headers; this renames them to safe placeholders so pandas can handle them

- **`extract_all_tables()`** — reads every table from every page of a PDF, one
  DataFrame per table. We use **looser table detection settings** to reduce the
  left-truncation problem we observed in earlier attempts.

- **`guess_sector()`** — classifies a sentence or phrase into a sector
  (Education, Health, Water, Roads, etc.) based on keyword matching

These utilities are used by every extraction stage in Phase 2.

In [29]:
# ============================================================
# PHASE 2 · CELL 2 (v4 — FINAL) — EXTRACTION UTILITIES
# ============================================================

def make_unique_columns(cols):
    """Return unique, non-empty column labels."""
    seen, out = {}, []
    for i, c in enumerate(cols):
        name = str(c).strip() if c is not None else ""
        if not name or name.lower() in ("nan", "none"):
            name = f"col_{i}"
        if name in seen:
            seen[name] += 1
            name = f"{name}__{seen[name]}"
        else:
            seen[name] = 0
        out.append(name)
    return out


def extract_all_tables(pdf_path):
    """Extract every table using the 'lines' strategy.

    Our diagnostic showed that:
      - 'lines' strategy → produces correct headers
      - 'text'  strategy → splits titles into fragments (BAD)
      - 'default'        → same as lines for this PDF

    So we always use 'lines'.
    """
    LINE_SETTINGS = {
        "vertical_strategy":   "lines",
        "horizontal_strategy": "lines",
    }
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables(LINE_SETTINGS):
                if tbl and len(tbl) > 1:
                    cols = make_unique_columns(tbl[0])
                    df = pd.DataFrame(tbl[1:], columns=cols)
                    df["_page"] = pno
                    tables.append(df)
    return tables


SECTOR_KEYWORDS = {
    "Education":            ["education", "school", "classroom", "teacher",
                             "pupil", "literacy", "desk"],
    "Health":               ["health", "clinic", "hospital", "nurse",
                             "doctor", "maternal", "malaria", "hiv"],
    "Water and Sanitation": ["water", "sanitation", "borehole", "toilet",
                             "latrine", "sewer", "sewage"],
    "Roads and Drainages":  ["road", "drain", "street", "bridge",
                             "culvert", "pavement", "tarmac"],
    "Commerce":             ["market", "trade", "commerce", "business",
                             "sme", "entrepreneur"],
    "Agriculture":          ["agriculture", "farm", "crop", "livestock",
                             "irrigation", "maize"],
    "Energy":               ["electricity", "power", "solar", "grid",
                             "energy", "zesco"],
    "Governance":           ["governance", "council", "ward", "community",
                             "participation", "committee"],
    "Environment":          ["environment", "climate", "forest", "tree",
                             "green", "pollution", "waste"],
}


def guess_sector(text):
    if not isinstance(text, str):
        return "Unknown"
    s = text.lower()
    scores = {sec: sum(1 for kw in kws if kw in s)
              for sec, kws in SECTOR_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Unknown"


print("Extraction utilities ready (lines strategy — validated).")

Extraction utilities ready (lines strategy — validated).


## Phase 2 · Section 2.2 — IDP Strategic Areas

The Kabwe IDP is anchored on the four strategic development areas from Zambia's
Eighth National Development Plan (8NDP):

- Economic Transformation and Job Creation
- Human and Social Development
- Environmental Sustainability
- Good Governance Environment

We extract each area name along with the pages where it appears and the IDP's
vision and mission statements. These four rows anchor the entire dataset — they
contextualise everything else.

In [33]:
# ============================================================
# PHASE 2 · CELL 2 — LOAD PDFs AS TEXT
# ============================================================

def load_pdf_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            pages.append((pno, page.extract_text() or ""))
    return pages


idp_pages     = load_pdf_pages(RAW_DIR / sources_by_id["IDP_MAIN"]["filename"])
citizen_pages = load_pdf_pages(RAW_DIR / sources_by_id["IDP_CITIZEN"]["filename"])
cdf_pages     = load_pdf_pages(RAW_DIR / sources_by_id["CDF_PROJECTS_2025"]["filename"])
news_pages    = load_pdf_pages(RAW_DIR / sources_by_id["NEWSLETTER_2024"]["filename"])

In [32]:
# ============================================================
# PHASE 2 · CELL 3 — IDP STRATEGIC AREAS
# ============================================================

STRATEGIC_AREAS = [
    "Economic Transformation and Job Creation",
    "Human and Social Development",
    "Environmental Sustainability",
    "Good Governance Environment",
]


def find_statement(text, keywords, context_lines=4, max_len=400):
    """Return the first text block following a keyword match."""
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if any(kw in line.lower() for kw in keywords):
            snippet = " ".join(lines[i:i+context_lines])
            return re.sub(r"\s+", " ", snippet).strip()[:max_len]
    return None


idp_text = "\n".join(t for _, t in idp_pages)
vision   = find_statement(idp_text, ["vision", "our vision"])
mission  = find_statement(idp_text, ["mission", "our mission"])

strategic_df = pd.DataFrame([
    {
        "strategic_area":  area,
        "page_refs":       ",".join(str(p) for p, t in idp_pages
                                     if area.lower() in t.lower()),
        "vision_excerpt":  vision,
        "mission_excerpt": mission,
        "source_doc":      "IDP_MAIN",
    }
    for area in STRATEGIC_AREAS
])

strategic_df.to_csv(EXTRACTED_DIR / "strategic_areas.csv", sep="|", index=False)
print(f"Strategic areas: {len(strategic_df)}")
strategic_df[["strategic_area", "page_refs"]]

Strategic areas: 4


,strategic_area,page_refs
0,Economic Transformation and Job Creation,"11,12,13,19,78,164,165,169,170,232,233,299"
1,Human and Social Development,"8,11,12,19,164,165,182,249,316"
2,Environmental Sustainability,"8,11,12,13,19,90,164,165,200,280,340"
3,Good Governance Environment,"11,13,165,205,285"


## Phase 2 · Section 2.3 — IDP Sector Goals

The IDP describes development goals in narrative form. These sentences usually
contain a target verb ("increase", "improve", "achieve") and sometimes a
quantified target ("from 78% to 92%", "by 2030").

We use regex patterns to find these sentences and assign each one a sector
label. The `page` column lets us trace every goal back to its source.

This extraction typically yields 40–150 goal statements.

In [34]:
# ============================================================
# PHASE 2 · CELL 4 — IDP SECTOR GOALS
# ============================================================

GOAL_PATTERNS = [
    r"\b(increase|reduce|expand|improve|achieve|ensure|promote|enhance|strengthen)\b.{10,300}",
    r"\bobjectives?\s+\d+(?:\.\d+)*[:\s].{10,300}",
    r"\b(target|goal|aim)s?\s*[:\-]\s*.{10,300}",
    r"\bby\s+20\d{2}\b.{0,200}",
    r"\bfrom\s+[\d.,%]+\s+to\s+[\d.,%]+\b.{0,200}",
]


def extract_goals(pages, source_doc):
    rows = []
    for pno, text in pages:
        for s in re.split(r"(?<=[.!?])\s+", text):
            s_clean = re.sub(r"\s+", " ", s).strip()
            if not (40 <= len(s_clean) <= 500):
                continue
            if any(re.search(p, s_clean, re.IGNORECASE) for p in GOAL_PATTERNS):
                rows.append({
                    "page":       pno,
                    "sector":     guess_sector(s_clean),
                    "goal_text":  s_clean[:400],
                    "source_doc": source_doc,
                })
    return rows


goals_df = (
    pd.DataFrame(extract_goals(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["goal_text"])
      .reset_index(drop=True)
)
goals_df.to_csv(EXTRACTED_DIR / "goals.csv", sep="|", index=False)

print(f"Sector goals: {len(goals_df)}")
print(goals_df["sector"].value_counts().to_string())

Sector goals: 256
sector
Unknown                 82
Environment             40
Governance              29
Education               23
Water and Sanitation    21
Agriculture             20
Health                  17
Commerce                10
Roads and Drainages      8
Energy                   6


## Phase 2 · Section 2.4 — IDP Baseline Statistics

The IDP is full of quantitative facts about the district — *"45 health posts"*,
*"population of 245,000"*, *"120 km of roads"*. These form the district profile.

We extract them with a regex matching a number followed by a countable noun.
Each statistic keeps the surrounding sentence as context.

**Note:** We deliberately do **not** assign a sector to each statistic — the
keyword-based sector classification is unreliable for these short facts, and
guessing wrong is worse than leaving it blank.

In [ ]:
# ============================================================
# PHASE 2 · CELL 5 — IDP BASELINE STATISTICS
# ============================================================

STAT_UNITS = [
    "school", "schools", "clinic", "clinics", "hospital", "hospitals",
    "health post", "health posts", "borehole", "boreholes",
    "market", "markets", "road", "roads", "km", "kilometre", "kilometres",
    "household", "households", "population", "people", "residents",
    "ward", "wards", "teacher", "teachers", "nurse", "nurses",
    "pupil", "pupils", "desk", "desks", "toilet", "toilets",
    "latrine", "latrines", "plot", "plots",
]

UNIT_REGEX = "|".join(re.escape(u) for u in STAT_UNITS)
STAT_REGEX = re.compile(rf"\b(\d[\d,]*)\s+({UNIT_REGEX})\b", re.IGNORECASE)


def extract_statistics(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in STAT_REGEX.finditer(text):
            ctx = text[max(0, m.start()-120):m.end()+120]
            rows.append({
                "page":       pno,
                "value":      int(m.group(1).replace(",", "")),
                "unit":       m.group(2).lower(),
                "context":    re.sub(r"\s+", " ", ctx).strip()[:300],
                "source_doc": source_doc,
            })
    return rows


stats_df = (
    pd.DataFrame(extract_statistics(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["value", "unit", "context"])
      .reset_index(drop=True)
)
stats_df.to_csv(EXTRACTED_DIR / "statistics.csv", sep="|", index=False)

print(f"Baseline statistics: {len(stats_df)}")
print(stats_df["unit"].value_counts().head(10).to_string())

## Phase 2 · Section 2.5 — CDF Project Registry

The 2025 Approved Community Projects document contains the master list of CDF
projects for Kabwe Central. This is the core of the project-records dataset.

We extract every table from every page, then filter to keep only the ones that
look like project registries (multiple rows, content mentioning projects, wards,
or amounts). The result is saved raw — cleaning happens in Phase 3.

In [ ]:
# ============================================================
# PHASE 2 · CELL 6 (v6 — FINAL FINAL) — CDF PROJECT REGISTRY
# ============================================================

def is_project_table(df, min_rows=5, min_cols=6):
    """Accept ONLY tables that are real project registries."""
    if df.shape[0] < min_rows or df.shape[1] < min_cols:
        return False
    headers = [str(c).strip().upper() for c in df.columns]
    header_text = " ".join(headers)
    if "PROJECT NAME" not in header_text:
        return False
    if "SUB TOTAL" in header_text or "SUMMARY" in header_text:
        return False
    if len(df) > 0:
        first_cell = str(df.iloc[0, 0]).strip().upper()
        if first_cell.startswith("SUB TOTAL") or first_cell.startswith("SUMMARY"):
            return False
    return True


# ---------- Extract ----------
cdf_tables = extract_all_tables(
    RAW_DIR / sources_by_id["CDF_PROJECTS_2025"]["filename"]
)

print(f"Tables detected: {len(cdf_tables)}")
for t in cdf_tables:
    print(f"  page {t['_page'].iloc[0]}: {t.shape}  "
          f"cols={list(t.columns)[:6]}")

# ---------- Filter ----------
main_tables = [t for t in cdf_tables if is_project_table(t)]
print(f"\nProject tables passing filter: {len(main_tables)}")

if not main_tables:
    print("No project tables matched — using all tables.")
    main_tables = cdf_tables

# ---------- Concatenate ----------
cdf_raw = (
    pd.concat(main_tables, ignore_index=True, sort=False)
    if main_tables else pd.DataFrame()
)

# ---------- Drop columns that are empty OR all-blank ----------
def is_blank(series):
    """True if every value is NaN, None, or an empty/whitespace string."""
    return series.apply(
        lambda v: pd.isna(v) or str(v).strip() == ""
    ).all()

before_cols = list(cdf_raw.columns)
drop_cols = [c for c in cdf_raw.columns if is_blank(cdf_raw[c])]

if drop_cols:
    cdf_raw = cdf_raw.drop(columns=drop_cols)
    print(f"\n  Dropped {len(drop_cols)} empty/blank columns: {drop_cols}")
else:
    print("\n  No empty columns to drop.")

# ---------- Metadata ----------
cdf_raw["source_doc"] = "CDF_PROJECTS_2025"

# ---------- Save ----------
cdf_raw.to_csv(EXTRACTED_DIR / "cdf_projects_raw.csv", sep="|", index=False)

print(f"\n CDF raw: {len(cdf_raw)} rows, {cdf_raw.shape[1]} cols")
print(f"   Columns: {list(cdf_raw.columns)}")
cdf_raw.head(10)

## Phase 2 · Section 2.6 — Newsletter Completed Projects

The 2024 Newsletter lists completed education and health projects from 2022
and 2023. This document uses a magazine-style layout that `pdfplumber`'s table
extraction can't parse reliably — the grid lines don't align with the logical
data cells.

So we use a different strategy: we scan the raw page text with a regex that
matches `<project name> ... <amount>` patterns, and filter for lines containing
project-related keywords.

In [ ]:
# ============================================================
# PHASE 2 · CELL 7 — NEWSLETTER PROJECTS
# ============================================================

PROJECT_KEYWORDS = [
    "school", "classroom", "block", "toilet", "ablution", "clinic",
    "market", "borehole", "desk", "chair", "construction", "procurement",
    "rehabilitation", "installation", "supply", "road", "bridge",
    "hospital", "office", "staff house", "shelter", "wall fence",
    "water", "reticulated", "solar", "panel",
]

NEWS_LINE = re.compile(
    r"(?P<name>(?:[A-Z][A-Za-z0-9'\-]+[\s,&]+){3,20}[A-Za-z0-9'\-]+)"
    r"[^\d]{0,80}(?:K|ZMW)?\s*"
    r"(?P<amount>\d{1,3}(?:,\d{3})+(?:\.\d{2})?)",
    re.MULTILINE,
)
YEAR_RE = re.compile(r"\b(20\d{2})\b")


def extract_newsletter(pages):
    rows = []
    for pno, text in pages:
        if len(text) < 200:
            continue
        for m in NEWS_LINE.finditer(text):
            name = re.sub(r"\s+", " ", m.group("name")).strip()
            if not (20 <= len(name) <= 200):
                continue
            if not any(kw in name.lower() for kw in PROJECT_KEYWORDS):
                continue
            try:
                amount = float(m.group("amount").replace(",", ""))
            except ValueError:
                continue
            if not (5_000 <= amount <= 500_000_000):
                continue
            ctx = text[max(0, m.start()-250):m.end()+250]
            ym  = YEAR_RE.search(ctx)
            rows.append({
                "project_name":        name[:200],
                "approved_amount_zmw": amount,
                "year_funded":         int(ym.group(1)) if ym else 2024,
                "sector":              guess_sector(name),
                "status":              "Completed",
                "source_doc":          "NEWSLETTER_2024",
                "page":                pno,
            })
    return rows


newsletter_df = (
    pd.DataFrame(extract_newsletter(news_pages))
      .drop_duplicates(subset=["project_name", "approved_amount_zmw"])
      .reset_index(drop=True)
)
newsletter_df.to_csv(EXTRACTED_DIR / "newsletter_raw.csv", sep="|", index=False)

print(f"Newsletter: {len(newsletter_df)} rows")
if not newsletter_df.empty:
    print(f"   Total ZMW: {newsletter_df['approved_amount_zmw'].sum():,.2f}")

## Phase 2 · Section 2.7 — Newsletter Narrative Sentences

The Newsletter also contains narrative articles that mention projects in prose
— with names, completion events, and numbers (amounts, counts, years).

We mine the raw page text for sentences that contain both a project keyword
and a number. Each sentence becomes a record, with the numbers extracted into
a separate column for easy scanning.

This complements the structured project lines by capturing context the tables
don't carry.

In [ ]:
# ============================================================
# PHASE 2 · CELL 8 — NEWSLETTER NARRATIVE SENTENCES
# ============================================================

NARRATIVE_KEYWORDS = [
    "school", "classroom", "clinic", "hospital", "market", "borehole",
    "desk", "water", "road", "bridge", "project", "handover",
    "commissioned", "completed", "constructed", "rehabilitated",
    "beneficiaries", "pupils", "patients", "residents",
]

NARRATIVE_NUMBER_RE = re.compile(r"\d[\d,]*")


def extract_narrative_sentences(pages, source_doc):
    rows = []
    for pno, text in pages:
        for s in re.split(r"(?<=[.!?])\s+", text):
            s_clean = re.sub(r"\s+", " ", s).strip()
            if not (50 <= len(s_clean) <= 500):
                continue
            lower = s_clean.lower()
            if not any(kw in lower for kw in NARRATIVE_KEYWORDS):
                continue
            if not NARRATIVE_NUMBER_RE.search(s_clean):
                continue
            numbers = [int(m.group().replace(",", ""))
                       for m in NARRATIVE_NUMBER_RE.finditer(s_clean)]
            rows.append({
                "page":          pno,
                "sector":        guess_sector(s_clean),
                "sentence":      s_clean[:400],
                "numbers_found": ",".join(map(str, numbers[:10])),
                "source_doc":    source_doc,
            })
    return rows


narrative_df = (
    pd.DataFrame(extract_narrative_sentences(news_pages, "NEWSLETTER_2024"))
      .drop_duplicates(subset=["sentence"])
      .reset_index(drop=True)
)
narrative_df.to_csv(EXTRACTED_DIR / "newsletter_narrative.csv", sep="|", index=False)

print(f"Narrative sentences: {len(narrative_df)}")
if not narrative_df.empty:
    print(f"\nSample:")
    for s in narrative_df["sentence"].head(5):
        print(f"  • {s[:100]}...")

## Phase 2 · Section 2.8 — IDP Sub-Programmes (Strict)

The IDP is organised around 8NDP pillars, and each pillar contains named
sub-programmes — the concrete interventions the council plans to undertake.

**A note on extraction difficulty:** this is the hardest extraction in the
pipeline. Numbered/bulleted lines in the IDP also appear in:

- The table of contents (`"3.1.4 Local Economic Development .... 22"`)
- The institutional arrangements chapter (`"Director of Planning John Doe"`)
- Staff lists (`"Health Officer Christopher Mtonga"`)
- Fragmentary headings (`"enabling environment for"`)

Our strict filter rejects all of these by requiring:

1. The line contains a **development action verb**
2. The line does **not** end with a preposition (rules out fragments)
3. The line does **not** match a person-name pattern (`Dr.`, `Officer Name`)
4. The line does **not** contain banned context phrases
5. The line has **4+ words** and 60%+ alphabetic characters

The result is a small, precise list of real sub-programmes — or an empty file
if the IDP presents them as prose rather than bullets.

In [ ]:
# ============================================================
# PHASE 2 · CELL 9 — IDP SUB-PROGRAMMES (STRICT)
# ============================================================

# ---------- Constants ----------
DOTTED_LEADER    = re.compile(r"\.{3,}")
ENDS_WITH_NUMBER = re.compile(r"\d\s*$")
TRAILING_PREPS   = re.compile(
    r"\b(?:for|of|in|to|with|and|by|at|on|the|a|an|from|into|as|is|are|was|were)\s*$",
    re.IGNORECASE,
)

ACTION_VERBS = [
    "construction", "construct", "rehabilitation", "rehabilitate",
    "expansion", "expand", "improvement", "improve", "promotion", "promote",
    "provision", "provide", "development", "develop", "establishment",
    "establish", "strengthening", "strengthen", "enhancement", "enhance",
    "maintenance", "maintain", "installation", "install", "supply",
    "upgrade", "upgrading", "extension", "extend", "creation", "create",
    "increase", "reduction", "reduce", "modernisation", "modernise",
    "replacement", "replace",
]

BANNED_CONTEXT = [
    "development plan", "citizens version", "central province",
    "district council", "urban council", "municipal council",
    "ward development committee", "planning unit", "ppu",
    "administrative", "implementing", "beneficiaries",
]

PERSON_NAME_PATTERN = re.compile(
    r"\b(?:Dr|Mr|Mrs|Ms|Miss|Eng|Prof)\.?\s+[A-Z][a-z]+",
    re.IGNORECASE,
)
TITLE_BEFORE_NAME = re.compile(
    r"(?:Officer|Director|Head|Manager|Specialist|Inspector|Planner|"
    r"Secretary|Administrator|Accountant|Engineer|Coordinator)\s+"
    r"[A-Z][a-z]+",
)


def is_real_sub_programme(line):
    """Strict filter for actual development sub-programmes."""
    if not isinstance(line, str):
        return False
    line = line.strip()

    # Length and shape
    if len(line) < 20 or len(line) > 180:
        return False
    if len(line.split()) < 4:
        return False
    if not line[0].isupper():
        return False

    # Not a TOC or fragment
    if DOTTED_LEADER.search(line):
        return False
    if ENDS_WITH_NUMBER.search(line):
        return False
    if TRAILING_PREPS.search(line):
        return False

    # Contains a development action verb
    lower = line.lower()
    if not any(verb in lower for verb in ACTION_VERBS):
        return False

    # Not institutional/banned text
    if any(bc in lower for bc in BANNED_CONTEXT):
        return False

    # Not a person name or title+name
    if PERSON_NAME_PATTERN.search(line):
        return False
    if TITLE_BEFORE_NAME.search(line):
        return False

    # At least 60% alphabetic
    alpha = sum(1 for c in line if c.isalpha())
    if alpha / len(line) < 0.6:
        return False

    return True


# ---------- Sub-programme extraction ----------
PILLAR_KEYWORDS = [
    "economic transformation", "job creation",
    "human and social development",
    "environmental sustainability", "good governance",
]

SUB_PROG_PATTERN = re.compile(
    r"^\s*(?:\d+(?:\.\d+)*|[•\-*]|[a-z]\))\s*(.{15,180})$",
    re.MULTILINE,
)


def extract_subprogrammes(pages):
    rows = []
    current_pillar = "Unknown"
    for pno, text in pages:
        for kw in PILLAR_KEYWORDS:
            if kw in text.lower():
                current_pillar = kw.title()
                break
        for m in SUB_PROG_PATTERN.finditer(text):
            line = re.sub(r"\s+", " ", m.group(1)).strip()
            if not is_real_sub_programme(line):
                continue
            rows.append({
                "page":           pno,
                "pillar":         current_pillar,
                "sub_programme":  line[:200],
                "source_doc":     "IDP_MAIN",
            })
    return rows


pillars_df = (
    pd.DataFrame(extract_subprogrammes(idp_pages))
      .drop_duplicates(subset=["sub_programme"])
      .reset_index(drop=True)
)
pillars_df.to_csv(EXTRACTED_DIR / "subprogrammes.csv", sep="|", index=False)

print(f"Sub-programmes (strict): {len(pillars_df)}")
if not pillars_df.empty:
    print("\nBy pillar:")
    print(pillars_df["pillar"].value_counts().to_string())
    print("\nSample:")
    print(pillars_df.head(15).to_string(index=False))
else:
    print("No real sub-programmes found — the IDP may present them as prose.")

## Phase 2 · Section 2.9 — IDP M&E Framework

The IDP's monitoring and evaluation framework lists every indicator the council
tracks against its goals. Each row typically has an indicator name, a baseline,
a target, and a data source.

We scan every IDP page for tables whose header row mentions indicator-related
keywords, then extract each row as a record.

In [ ]:
# ============================================================
# PHASE 2 · CELL 10 — IDP M&E FRAMEWORK
# ============================================================

ME_KEYWORDS = (
    "indicator", "baseline", "target", "means of verification",
    "data source", "frequency", "responsible", "outcome",
    "output indicator", "performance", "strategies", "program",
    "activities", "location",
)


def extract_me_framework(pdf_path):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables():
                if not tbl or len(tbl) < 2:
                    continue
                header_text = " ".join(str(c or "").lower() for c in tbl[0])
                kw_hits = sum(1 for kw in ME_KEYWORDS if kw in header_text)
                if kw_hits < 2:
                    continue
                cols = [str(c or f"col_{i}").strip()[:60]
                        for i, c in enumerate(tbl[0])]
                for r in tbl[1:]:
                    if not r or not any(r):
                        continue
                    record = {"page": pno, "source_doc": "IDP_MAIN"}
                    for c, v in zip(cols, r):
                        record[c] = str(v or "").strip()[:300]
                    rows.append(record)
    return rows


me_raw = extract_me_framework(RAW_DIR / sources_by_id["IDP_MAIN"]["filename"])

me_df = (
    pd.DataFrame(me_raw)
      .drop_duplicates()
      .reset_index(drop=True)
    if me_raw else pd.DataFrame()
)
me_df.to_csv(EXTRACTED_DIR / "me_framework.csv", sep="|", index=False)

print(f"M&E framework rows: {len(me_df)}")
if not me_df.empty:
    print(f"   Columns: {list(me_df.columns)}")
    me_df.head()

## Phase 2 · Section 2.10 — IDP Sector Cost Estimates

The IDP contains cost estimate tables for each sector — columns are years
(2023, 2024, …, 2033), rows are programmes. We reshape this from wide to long
form so each cell becomes a `(programme, year, cost)` record.

We scan every page for tables whose header contains at least two distinct years.

In [ ]:
# ============================================================
# PHASE 2 · CELL 11 — IDP SECTOR COST ESTIMATES
# ============================================================

YEAR_TOKENS = [str(y) for y in range(2023, 2034)]


def extract_cost_tables(pdf_path):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables():
                if not tbl or len(tbl) < 3:
                    continue
                header = " ".join(str(c or "") for c in tbl[0])
                year_hits = sum(1 for y in YEAR_TOKENS if y in header)
                if year_hits < 2:
                    continue
                for r in tbl[1:]:
                    if not r or len(r) < 3:
                        continue
                    label = str(r[0] or "").strip()
                    if len(label) < 4:
                        continue
                    for h, v in zip(tbl[0][1:], r[1:]):
                        year_str = str(h or "").strip()
                        if not year_str.isdigit():
                            continue
                        try:
                            cost = float(
                                str(v).replace(",", "")
                                      .replace("K", "")
                                      .replace("ZMW", "")
                                      .strip() or 0
                            )
                        except ValueError:
                            continue
                        if cost <= 0:
                            continue
                        rows.append({
                            "programme":  label[:120],
                            "year":       int(year_str),
                            "cost_zmw":   cost,
                            "page":       pno,
                            "source_doc": "IDP_MAIN",
                        })
    return rows


cost_df = (
    pd.DataFrame(extract_cost_tables(RAW_DIR / sources_by_id["IDP_MAIN"]["filename"]))
      .drop_duplicates(subset=["programme", "year", "cost_zmw"])
      .reset_index(drop=True)
)
cost_df.to_csv(EXTRACTED_DIR / "cost_estimates.csv", sep="|", index=False)

print(f"Cost estimate rows: {len(cost_df)}")
if not cost_df.empty:
    print(f"   Unique programmes: {cost_df['programme'].nunique()}")
    print(f"   Years: {sorted(cost_df['year'].unique())}")
    cost_df.head()

## Phase 2 · Section 2.11 — IDP Ward Profiles

The IDP contains ward-level descriptions — names, numbers, and sometimes
population figures or dominant economic activity.

We extract ward names and numbers using a pattern that matches
`"Ward <number>: <name>"` and surrounding context.

In [ ]:
# ============================================================
# PHASE 2 · CELL 12 — IDP WARD PROFILES
# ============================================================

WARD_PATTERN = re.compile(
    r"ward\s+(\d+)\s*[:\-]?\s*([A-Za-z][A-Za-z\s\-']{2,60})",
    re.IGNORECASE,
)


def extract_wards(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in WARD_PATTERN.finditer(text):
            wno   = m.group(1).strip()
            wname = re.sub(r"\s+", " ", m.group(2)).strip()
            if len(wname) < 3:
                continue
            if wname.lower() in ("development", "committee", "council",
                                 "the", "a", "an"):
                continue
            rows.append({
                "ward_number": wno,
                "ward_name":   wname[:80],
                "page":        pno,
                "source_doc":  source_doc,
            })
    return rows


wards_df = (
    pd.DataFrame(extract_wards(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["ward_number", "ward_name"])
      .reset_index(drop=True)
)
wards_df.to_csv(EXTRACTED_DIR / "wards.csv", sep="|", index=False)

print(f"Ward profiles: {len(wards_df)}")
if not wards_df.empty:
    wards_df.head(10)

## Phase 2 · Section 2.12 — Citizen IDP Points

The Citizen IDP is a shorter, community-facing version of the main IDP. It
contains distilled priority statements that don't always appear verbatim in the
main document.

We extract numbered/bulleted lines that contain real development content, using
the same strict filter as the sub-programme extractor.

In [ ]:
# ============================================================
# PHASE 2 · CELL 13 — CITIZEN IDP POINTS
# ============================================================

def extract_citizen_points(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in SUB_PROG_PATTERN.finditer(text):
            line = re.sub(r"\s+", " ", m.group(1)).strip()
            if not is_real_sub_programme(line):
                continue
            rows.append({
                "page":       pno,
                "topic":      guess_sector(line),
                "point":      line[:300],
                "source_doc": source_doc,
            })
    return rows


citizen_df = (
    pd.DataFrame(extract_citizen_points(citizen_pages, "IDP_CITIZEN"))
      .drop_duplicates(subset=["point"])
      .reset_index(drop=True)
)
citizen_df.to_csv(EXTRACTED_DIR / "citizen_points.csv", sep="|", index=False)

print(f"Citizen points: {len(citizen_df)}")
if not citizen_df.empty:
    citizen_df.head(10)
else:
    print("No real citizen points found.")

## Phase 2 · Section 2.13 — Extraction Summary

We list every file we produced in `data/extracted/` — these are the inputs to
Phase 3.

In [ ]:
# ============================================================
# PHASE 2 · CELL 14 — EXTRACTION SUMMARY
# ============================================================

print("=" * 72)
print("PHASE 2 — EXTRACTION SUMMARY")
print("=" * 72)

extracted_files = sorted(EXTRACTED_DIR.glob("*.csv"))
total_rows = 0
for f in extracted_files:
    try:
        df = pd.read_csv(f, sep="|")
        total_rows += len(df)
        print(f"  {f.name:<40}  {len(df):>5} rows  {df.shape[1]:>2} cols")
    except Exception as e:
        print(f"  {f.name:<40}  [error] {e}")

print("-" * 72)
print(f"  {'TOTAL':<40}  {total_rows:>5} rows")
print(f"\n  Files in: {EXTRACTED_DIR.resolve()}")
print("  → Phase 2 complete. Proceed to Phase 3.")

# PHASE 3 — CLEANING & PREPROCESSING

## Objective

We now transform the raw extracted data into an analysis-ready dataset. This is
the phase where we:

1. Standardise column names
2. Clean text values (strip newlines, collapse whitespace)
3. Coerce amount fields to numeric
4. Canonicalise categorical fields
5. Drop header-repeat rows and junk
6. Merge into a master project table
7. Enrich with derived analytics columns
8. Build derived analytics tables (ward matrix, timeline)
9. Export pipe-delimited CSVs to `outputs/`

We keep cleaning separate from extraction because cleaning rules change more
often than extraction logic — and running cleaning on its own is much faster
than re-parsing the PDFs.

## Phase 3 · Section 3.0 — Setup

We reload the extracted tables from disk and re-establish the folder paths.
This cell makes Phase 3 self-contained: if we ever want to run Phase 3 without
re-running Phases 1 and 2, we can jump straight here.

The `extracted` dictionary gives us named access to each of the raw tables
we'll be cleaning.

In [ ]:
# ============================================================
# PHASE 3 · CELL 1 — SETUP
# ============================================================

import os
import re
import json
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=Warning)

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

EXTRACTED_DIR = REPO_ROOT / "data" / "extracted"
CLEAN_DIR     = REPO_ROOT / "data" / "clean"
OUTPUT_DIR    = REPO_ROOT / "outputs"

for d in (CLEAN_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- Project constants ----------
PROJECT_CODE = "db-unza26-csc4792"
COUNCIL_SLUG = "kabwe"
COUNCIL_FULL = "Kabwe Municipal Council"
CONSTITUENCY = "Kabwe Central"
RUN_STAMP    = datetime.now().isoformat(timespec="seconds")

# ---------- Load all extracted CSVs into a dict ----------
extracted = {}
if EXTRACTED_DIR.exists():
    for f in sorted(EXTRACTED_DIR.glob("*.csv")):
        try:
            extracted[f.stem] = pd.read_csv(f, sep="|")
        except Exception as e:
            print(f"  [warn] could not load {f.name}: {e}")

print("PHASE 3 — CLEANING & PREPROCESSING")
print(f"  Extracted dir : {EXTRACTED_DIR.resolve()}")
print(f"  Clean dir     : {CLEAN_DIR.resolve()}")
print(f"  Output dir    : {OUTPUT_DIR.resolve()}")
print(f"\n  Loaded {len(extracted)} extracted tables:")
for name, df in extracted.items():
    print(f"    {name:<30}  {df.shape}")

## Phase 3 · Section 3.1 — Cleaning Utilities

Before touching any individual table, we define the shared cleaning functions
that every downstream stage will use.

These are the workhorses of Phase 3:

- **`clean_text()`** — strips line breaks, collapses whitespace, drops null-like
  strings
- **`clean_amount()`** — converts currency strings to `float`
- **`canonical_sector()`** — maps any sector variant to a canonical name
- **`canonical_ward()`** — normalises ward names
- **`make_project_ids()`** — generates sequential primary keys
- **`is_real_item()`** — our junk filter for text-heavy extractions; rejects
  TOC entries (`"......"`), page-number fragments, mid-word truncations, and
  lines without a domain signal word

In [ ]:
# ============================================================
# PHASE 3 · CELL 2 — CLEANING UTILITIES
# ============================================================

NULL_STRINGS = {"none", "nan", "nat", "null", "n/a", "", "-", "unknown"}

VAGUE_WARDS = {
    "various wards", "all wards", "unknown", "none", "",
    "entire district", "all", "n/a",
}


# ---------- Text cleaning ----------
def clean_text(value, max_len=None):
    """Trim, collapse whitespace, and drop null-like strings."""
    if pd.isna(value):
        return None
    s = re.sub(r"[\r\n\t]+", " ", str(value))
    s = re.sub(r"\s+", " ", s).strip()
    if s.lower() in NULL_STRINGS:
        return None
    return s[:max_len] if max_len else s


# ---------- Amount cleaning ----------
def clean_amount(value):
    """Convert 'K 1,703,868.67' to 1703868.67. Returns None on failure."""
    if pd.isna(value):
        return None
    s = str(value).upper()
    s = re.sub(r"(ZMW|K|MK|KWACHA)", "", s)
    s = re.sub(r"[^\d.]", "", s)
    try:
        return float(s) if s else None
    except ValueError:
        return None


# ---------- Sector canonicalisation ----------
SECTOR_MAP = {
    "education":            "Education",
    "school":               "Education",
    "schools":              "Education",
    "health":               "Health",
    "clinic":               "Health",
    "hospital":             "Health",
    "water and sanitation": "Water and Sanitation",
    "water & sanitation":   "Water and Sanitation",
    "water":                "Water and Sanitation",
    "sanitation":           "Water and Sanitation",
    "roads and drainages":  "Roads and Drainages",
    "roads & drainages":    "Roads and Drainages",
    "roads":                "Roads and Drainages",
    "drainage":             "Roads and Drainages",
    "commerce":             "Commerce",
    "market":               "Commerce",
    "trade":                "Commerce",
    "agriculture":          "Agriculture",
    "farming":              "Agriculture",
    "energy":               "Energy",
    "electricity":          "Energy",
    "governance":           "Governance",
    "environment":          "Environment",
    "housing":              "Housing",
    "community development": "Community Development",
}


def canonical_sector(value):
    if pd.isna(value):
        return "Unknown"
    return SECTOR_MAP.get(str(value).strip().lower(),
                          str(value).strip().title())


# ---------- Ward canonicalisation ----------
def canonical_ward(value):
    if pd.isna(value):
        return None
    s = re.sub(r"\s+", " ", str(value)).strip().strip(",.")
    return s.title() if s else None


# ---------- ID generation ----------
def make_project_ids(n, prefix="KAB"):
    return [f"{prefix}-{i:04d}" for i in range(1, n + 1)]


# ---------- Junk filter ----------
DOTTED_LEADER     = re.compile(r"\.{3,}")
ENDS_WITH_NUMBER  = re.compile(r"\d\s*$")

SUB_PROG_SIGNALS = [
    "development", "management", "promotion", "provision", "expansion",
    "improvement", "enhancement", "strengthening", "creation",
    "construction", "rehabilitation", "maintenance", "planning",
    "regulation", "monitoring", "coordination", "capacity",
    "infrastructure", "services", "access", "delivery",
    "investment", "agriculture", "education", "health",
    "water", "sanitation", "roads", "energy", "markets",
    "forestry", "environment", "climate", "tourism", "commerce",
    "trade", "industry", "housing", "settlement", "land",
    "governance", "administration", "finance", "revenue",
]


def is_real_item(line, min_words=3, require_signal=True):
    """Return True if a candidate line is NOT a TOC entry or fragment."""
    if not isinstance(line, str):
        return False
    if DOTTED_LEADER.search(line):
        return False
    if ENDS_WITH_NUMBER.search(line):
        return False
    if len(line.split()) < min_words:
        return False
    if line.isupper() and len(line) > 60:
        return False
    if require_signal:
        lower = line.lower()
        if not any(sig in lower for sig in SUB_PROG_SIGNALS):
            return False
    return True


print("Cleaning utilities ready.")

## Phase 3 · Section 3.2 — Clean the CDF Project Registry

The CDF project registry is our core dataset. We apply the full cleaning pipeline:

1. **Rename columns** to a canonical set (uppercase variants, common spelling
   differences, and truncated PDF headers all map to the same target)
2. **Coalesce duplicate columns** if the PDF split one logical column
3. **Drop stray columns** (empty placeholders, page markers, autogenerated)
4. **Clean text fields** (project name, ward, sector, site, description)
5. **Clean amounts** to `float`
6. **Drop header-repeat rows** — PDFs repeat the header on every page
7. **Drop rows without a valid project name**
8. **Drop "Sub Total" rows** and rows with no amount
9. **Fill missing sectors and wards** with `"Unknown"` so group-bys don't break
10. **Add metadata columns** (council, constituency, status, source)

The result is saved to `data/clean/projects_clean.csv`.

In [ ]:
# ============================================================
# PHASE 3 · CELL 3 — CLEAN CDF PROJECT REGISTRY
# ============================================================

cdf = extracted.get("cdf_projects_raw", pd.DataFrame()).copy()

if cdf.empty:
    print("cdf_projects_raw.csv is empty — check Phase 2.")
    cdf_clean = pd.DataFrame()
else:
    print(f"Raw CDF rows: {len(cdf)}")

    # 1. Uppercase columns
    cdf.columns = [str(c).strip().upper() for c in cdf.columns]

    # 2. Rename to canonical schema
    RENAME_MAP = {
        "NO.": "project_no", "NO": "project_no", "#": "project_no",
        "S/N": "project_no", "SN": "project_no",
        "PROJECT NAME": "project_name", "PROJECT_NAME": "project_name",
        "PROJECT": "project_name", "NAME": "project_name",
        "PROJECT TITLE": "project_name",
        "PROJECT DESCRIPTION": "description", "DESCRIPTION": "description",
        "SECTOR": "sector", "CATEGORY": "sector",
        "WARD": "ward", "LOCATION": "ward", "AREA": "ward",
        "PROJECT SITE": "site", "SITE": "site",
        "COL_5": "site",
        "ENGINEERS ESTIMATE": "engineer_estimate_zmw",
        "ENGINEER'S ESTIMATE": "engineer_estimate_zmw",
        "ENGINEER ESTIMATE": "engineer_estimate_zmw",
        "ENGINEERS ESTIMAT": "engineer_estimate_zmw",
        "ENGINEER ESTIMAT": "engineer_estimate_zmw",
        "ESTIMATE": "engineer_estimate_zmw",
        "APPROVED AMOUNT": "approved_amount_zmw",
        "APPROVED AMOUNT (ZMW)": "approved_amount_zmw",
        "APPROVED AMOUN": "approved_amount_zmw",
        "APPROVED AMOU": "approved_amount_zmw",
        "APPROVED AMT": "approved_amount_zmw",
        "AMOUNT": "approved_amount_zmw",
        "AMOUNT (ZMW)": "approved_amount_zmw",
        "COST": "approved_amount_zmw",
        "BUDGET": "approved_amount_zmw",
        "ESTIMATE__1": "approved_amount_zmw",
        "SOURCE_DOC": "source_doc",   # ← normalise to lowercase early
    }
    cdf = cdf.rename(columns=RENAME_MAP)

    # 3. Drop stray columns
    DROP_COLS = [c for c in cdf.columns
                 if c.startswith("COL_") or c == "_PAGE" or c == ""
                 or c.startswith("ESTIMATE__") or c.startswith("AMOUNT__")
                 or c.startswith("WARD__") or c.startswith("SECTOR__")]
    cdf = cdf.drop(columns=DROP_COLS, errors="ignore")
    print(f"After dropping stray columns: {cdf.shape}")

    # 4. Coalesce duplicate canonical columns
    def coalesce(df, col):
        positions = [i for i, c in enumerate(df.columns) if c == col]
        if len(positions) <= 1:
            return df
        combined = df.iloc[:, positions[0]]
        for pos in positions[1:]:
            combined = combined.where(combined.notna(), df.iloc[:, pos])
        df = df.drop(df.columns[positions], axis=1)
        df.insert(positions[0], col, combined)
        return df

    for canonical in ["project_name", "approved_amount_zmw",
                      "engineer_estimate_zmw", "ward", "sector",
                      "description", "site", "project_no", "source_doc"]:
        cdf = coalesce(cdf, canonical)

    # 5. Clean text
    for c in ["project_name", "description", "sector", "ward", "site"]:
        if c in cdf.columns:
            cdf[c] = cdf[c].apply(clean_text)

    # 6. Clean amounts
    for c in ["engineer_estimate_zmw", "approved_amount_zmw"]:
        if c in cdf.columns:
            cdf[c] = cdf[c].apply(clean_amount)

    # 7. Drop header repeats
    if "project_no" in cdf.columns:
        cdf = cdf[
            ~cdf["project_no"].astype(str).str.upper()
              .str.contains("PROJECT|NAME|WARD|SECTOR|AMOUNT",
                            na=False, regex=True)
        ]

    # 8. Drop rows without a valid project name
    if "project_name" in cdf.columns:
        cdf = cdf[
            cdf["project_name"].notna()
            & (cdf["project_name"].astype(str).str.len() >= 5)
            & (~cdf["project_name"].str.lower().str.contains(
                "sub total|total|summary", na=False))
        ]

    # 9. Drop rows with no amount
    amt_cols = [c for c in ["engineer_estimate_zmw", "approved_amount_zmw"]
                if c in cdf.columns]
    if amt_cols:
        cdf = cdf[cdf[amt_cols].notna().any(axis=1)]

    # 10. Canonicalise
    if "sector" in cdf.columns:
        cdf["sector"] = cdf["sector"].apply(canonical_sector)
    if "ward" in cdf.columns:
        cdf["ward"] = cdf["ward"].apply(canonical_ward)

    # 11. Fill missing categorical values
    for c in ["sector", "ward", "site", "description"]:
        if c in cdf.columns:
            cdf[c] = cdf[c].fillna("Unknown")

    # 12. Drop any remaining duplicate uppercase SOURCE_DOC
    if "SOURCE_DOC" in cdf.columns:
        cdf = cdf.drop(columns=["SOURCE_DOC"])

    # 13. Metadata
    cdf["council"]        = COUNCIL_FULL
    cdf["constituency"]   = CONSTITUENCY
    cdf["status"]         = "Approved"
    cdf["funding_source"] = "CDF"
    cdf["source_doc"]     = "CDF_PROJECTS_2025"
    cdf["scraped_at"]     = RUN_STAMP

    # 14. Final order and save
    cdf = cdf.reset_index(drop=True)
    cdf_clean = cdf.copy()
    cdf_clean.to_csv(CLEAN_DIR / "projects_clean.csv", sep="|", index=False)

    print(f"\n✅ Cleaned CDF rows: {len(cdf_clean)}")
    print(f"   Columns: {list(cdf_clean.columns)}")
    cdf_clean.head()

## Phase 3 · Section 3.3 — Clean the Newsletter Projects

The Newsletter projects have a simpler schema. We apply the same cleaning
principles and align them to the master schema so they can be merged.

Because the Newsletter PDF has a magazine layout, project names may be
truncated at the left edge. We accept this as a limitation and document it in
the Data in Brief paper.

In [ ]:
# ============================================================
# PHASE 3 · CELL 4 — CLEAN NEWSLETTER PROJECTS
# ============================================================

news = extracted.get("newsletter_raw", pd.DataFrame()).copy()

if news.empty:
    print("newsletter_raw.csv is empty — skipping.")
    news_clean = pd.DataFrame()
else:
    print(f"Raw Newsletter rows: {len(news)}")

    # Clean text
    for c in ["project_name", "sector", "status"]:
        if c in news.columns:
            news[c] = news[c].apply(clean_text)

    # Amount
    if "approved_amount_zmw" in news.columns:
        news["approved_amount_zmw"] = news["approved_amount_zmw"].apply(clean_amount)

    # Canonicalise sector
    if "sector" in news.columns:
        news["sector"] = news["sector"].apply(canonical_sector)

    # Metadata
    news["council"]        = COUNCIL_FULL
    news["constituency"]   = CONSTITUENCY
    news["funding_source"] = "CDF"
    news["scraped_at"]     = RUN_STAMP

    # Drop rows without a valid project name
    if "project_name" in news.columns:
        news = news[
            news["project_name"].notna()
            & (news["project_name"].astype(str).str.len() >= 5)
        ]

    news_clean = news.reset_index(drop=True)
    news_clean.to_csv(CLEAN_DIR / "newsletter_clean.csv", sep="|", index=False)

    print(f"\nCleaned Newsletter rows: {len(news_clean)}")
    print(news_clean.head())

## Phase 3 · Section 3.4 — Clean the Narrative Tables

The remaining tables — strategic areas, goals, statistics, sub-programmes,
M&E framework, cost estimates, newsletter narrative, citizen points — carry
mostly text with some numeric fields.

For each one we:

1. Clean every text column (`clean_text`)
2. Drop junk rows using our `is_real_item` filter where applicable
3. Coerce numeric fields
4. Save each to `data/clean/`

The `drop_junk_rows()` function applies the shared junk filter to the
specific column that carries the primary content for that table.

In [ ]:
# ============================================================
# PHASE 3 · CELL 5 — CLEAN NARRATIVE TABLES
# ============================================================

TEXT_COL_FOR_JUNK = {
    "goals":                "goal_text",
    "subprogrammes":        "sub_programme",
    "me_framework":         "Activities",
    "citizen_points":       "point",
    "citizen_priorities":   "priority",
    "newsletter_narrative": "sentence",
    "strategic_areas":      "strategic_area",
    "wards":                "ward_name",
    "statistics":           "context",
    "cost_estimates":       "programme",
}


def drop_junk_rows(df, text_col, min_words=4):
    """Remove rows where the target text column looks like junk."""
    if df.empty or text_col not in df.columns:
        return df
    mask = df[text_col].apply(
        lambda s: isinstance(s, str) and is_real_item(s, min_words=min_words)
    )
    return df[mask].reset_index(drop=True)


cleaned_summary = []
narrative_keys = [
    "strategic_areas",
    "goals",
    "statistics",
    "subprogrammes",
    "me_framework",
    "cost_estimates",
    "newsletter_narrative",
    "citizen_points",
    "citizen_priorities",
    "wards",
]

print(f"{'Table':<25} {'Before':>8} {'After':>8}")
print("-" * 45)

for key in narrative_keys:
    df = extracted.get(key, pd.DataFrame()).copy()
    if df.empty:
        cleaned_summary.append((key, 0, 0))
        print(f"{key:<25} {'(empty)':>8} {'-':>8}")
        continue

    rows_before = len(df)

    # Clean all text columns
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].apply(clean_text)

    # Apply junk filter where appropriate
    if key in TEXT_COL_FOR_JUNK:
        df = drop_junk_rows(df, TEXT_COL_FOR_JUNK[key])

    # Coerce numeric fields
    if key == "cost_estimates":
        if "year" in df.columns:
            df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        if "cost_zmw" in df.columns:
            df["cost_zmw"] = pd.to_numeric(df["cost_zmw"], errors="coerce")
    if key == "statistics" and "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df.to_csv(CLEAN_DIR / f"{key}_clean.csv", sep="|", index=False)
    cleaned_summary.append((key, rows_before, len(df)))
    print(f"{key:<25} {rows_before:>8} {len(df):>8}")

print("\nNarrative tables cleaned.")

## Phase 3 · Section 3.5 — Merge Into Master Project Table

We union the CDF projects and Newsletter projects into one table with a common
schema. Every row gets a synthetic `project_id` and carries provenance metadata
(`source_doc`, `funding_source`, `scraped_at`).

This is the master table that will be our headline Kaggle dataset.

In [ ]:
# ============================================================
# PHASE 3 · CELL 6 — MERGE INTO MASTER PROJECT TABLE
# ============================================================

PROJECT_COLUMNS = [
    "project_id", "council", "project_name", "sector", "constituency",
    "ward", "approved_amount_zmw", "status", "funding_source",
    "source_doc", "scraped_at",
]

frames = []
if not cdf_clean.empty:
    frames.append(cdf_clean.reindex(columns=PROJECT_COLUMNS))
if not news_clean.empty:
    frames.append(news_clean.reindex(columns=PROJECT_COLUMNS))

projects_master = (
    pd.concat(frames, ignore_index=True, sort=False)
    if frames
    else pd.DataFrame(columns=PROJECT_COLUMNS)
)

projects_master["project_id"] = make_project_ids(len(projects_master))
projects_master["approved_amount_zmw"] = pd.to_numeric(
    projects_master["approved_amount_zmw"], errors="coerce"
)

projects_master = projects_master.reset_index(drop=True)
projects_master.to_csv(CLEAN_DIR / "projects_master.csv", sep="|", index=False)

print(f"Master project table: {projects_master.shape}")
print(f"   Columns: {list(projects_master.columns)}")
print(projects_master.head())

## Phase 3 · Section 3.6 — Enrichment and Derived Columns

We add analytical columns that make the master table easier to use:

- **`sector_category`** — Social, Infrastructure, Economic, Governance, Other
- **`has_amount_zmw`** — boolean
- **`is_approved`** — boolean
- **`amount_band`** — Small, Medium, Large, Very Large
- **`has_ward`** — boolean (ward is not vague)

In [ ]:
# ============================================================
# PHASE 3 · CELL 7 — ENRICHMENT & DERIVED COLUMNS
# ============================================================

projects_master["has_amount_zmw"] = projects_master["approved_amount_zmw"].notna()

projects_master["is_approved"] = (
    projects_master["status"].astype(str).str.lower() == "approved"
)

SECTOR_GROUP = {
    "Education":            "Social",
    "Health":               "Social",
    "Water and Sanitation": "Infrastructure",
    "Roads and Drainages":  "Infrastructure",
    "Commerce":             "Economic",
    "Agriculture":          "Economic",
    "Energy":               "Infrastructure",
    "Governance":           "Governance",
    "Environment":          "Environment",
    "Housing":              "Infrastructure",
}
projects_master["sector_category"] = (
    projects_master["sector"].map(SECTOR_GROUP).fillna("Other")
)

def amount_band(a):
    if pd.isna(a):
        return "Unknown"
    if a < 100_000:
        return "Small (<100K)"
    if a < 1_000_000:
        return "Medium (100K-1M)"
    if a < 5_000_000:
        return "Large (1M-5M)"
    return "Very Large (>=5M)"

projects_master["amount_band"] = projects_master["approved_amount_zmw"].apply(amount_band)

projects_master["has_ward"] = (
    projects_master["ward"].notna()
    & (~projects_master["ward"].astype(str).str.lower().isin(VAGUE_WARDS))
)

projects_master.to_csv(CLEAN_DIR / "projects_master_enriched.csv",
                       sep="|", index=False)

print(f"Enriched: {projects_master.shape}")
print(f"   Columns: {list(projects_master.columns)}")
print("\nSector distribution:")
print(projects_master["sector"].value_counts().to_string())
print("\nAmount band distribution:")
print(projects_master["amount_band"].value_counts().to_string())

## Phase 3 · Section 3.7 — Deduplicate

We drop rows that describe the same project twice. Our dedup key is
`project_name | ward | approved_amount` — this catches genuine duplicates
without collapsing similar-sounding projects in different wards.

In [ ]:
# ============================================================
# PHASE 3 · CELL 8 — DEDUPLICATE
# ============================================================

before = len(projects_master)

key = (
    projects_master["project_name"].fillna("").str.lower().str.strip()
    + "|" + projects_master["ward"].fillna("").str.lower().str.strip()
    + "|" + projects_master["approved_amount_zmw"].fillna(-1).astype(str)
)

projects_master = (
    projects_master
      .assign(_key=key)
      .drop_duplicates(subset=["_key"], keep="first")
      .drop(columns=["_key"])
      .reset_index(drop=True)
)

projects_master["project_id"] = make_project_ids(len(projects_master))

after = len(projects_master)
print(f"Before dedup : {before}")
print(f"After dedup  : {after}")
print(f"Removed      : {before - after}")

projects_master.to_csv(CLEAN_DIR / "projects_master_final.csv",
                       sep="|", index=False)

## Phase 3 · Section 3.8 — Derived Analytics Tables

We build two derived analytics tables from the master project table:

- **Sector × Ward matrix** — projects per sector per ward (spatial view)
- **Timeline** — project count and total ZMW by funding year (temporal view)

In [ ]:
# ============================================================
# PHASE 3 · CELL 9 — DERIVED ANALYTICS TABLES
# ============================================================

# Sector × Ward matrix
sector_by_ward = (
    projects_master[projects_master["has_ward"] == True]
      .groupby(["ward", "sector"], dropna=False)
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
sector_by_ward.columns.name = None
sector_by_ward = sector_by_ward.rename(columns={"ward": "ward_name"})

sector_cols = [c for c in sector_by_ward.columns if c != "ward_name"]
sector_by_ward["total_projects"] = sector_by_ward[sector_cols].sum(axis=1)

sector_by_ward.to_csv(CLEAN_DIR / "sector_by_ward.csv", sep="|", index=False)
print(f"Sector × Ward matrix: {sector_by_ward.shape}")
print(sector_by_ward.head())

# Timeline
def infer_year(source_doc):
    if "NEWSLETTER" in str(source_doc):
        return 2024
    if "CDF_PROJECTS_2025" in str(source_doc):
        return 2025
    return None

projects_master["inferred_year"] = projects_master["source_doc"].apply(infer_year)

timeline = (
    projects_master
      .dropna(subset=["inferred_year"])
      .groupby("inferred_year", as_index=False)
      .agg(
          project_count=("project_id", "count"),
          total_amount_zmw=("approved_amount_zmw", "sum"),
      )
      .rename(columns={"inferred_year": "year"})
)

timeline.to_csv(CLEAN_DIR / "timeline.csv", sep="|", index=False)
print(f"\nTimeline: {timeline.shape}")
print(timeline.to_string(index=False))

## Phase 3 · Section 3.9 — Export Final CSVs

We export every cleaned table to `outputs/` following the required naming
convention: pipe-delimited CSVs with filenames of the form
`db-unza26-csc4792-kabwe_<description>.csv`.

Empty tables are skipped so we don't ship header-only files.

In [ ]:
# ============================================================
# PHASE 3 · CELL 10 (v3 — with citizen_points) — EXPORT
# ============================================================

def load_clean(name):
    p = CLEAN_DIR / f"{name}_clean.csv"
    return pd.read_csv(p, sep="|") if p.exists() else pd.DataFrame()


outputs = {
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_projects.csv":         projects_master,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_strategic_areas.csv":  load_clean("strategic_areas"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_goals.csv":            load_clean("goals"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_statistics.csv":       load_clean("statistics"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_subprogrammes.csv":    load_clean("subprogrammes"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_me_framework.csv":     load_clean("me_framework"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_cost_estimates.csv":   load_clean("cost_estimates"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_citizen_points.csv":   load_clean("citizen_points"),   # ← ADD THIS
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_newsletter_narrative.csv": load_clean("newsletter_narrative"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_sector_by_ward.csv":       sector_by_ward,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_timeline.csv":             timeline,
}

print("Writing CSVs to outputs/:\n")
written, skipped = 0, 0
for fname, df in outputs.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        df.to_csv(OUTPUT_DIR / fname, sep="|", index=False, encoding="utf-8")
        print(f"  [written] {fname:<60}  {len(df):>5} rows  {df.shape[1]:>2} cols")
        written += 1
    else:
        print(f"  [skip]    {fname:<60}  (empty)")
        skipped += 1

print(f"\n📊 {written} files written, {skipped} skipped")

## Phase 3 · Section 3.10 — Data Quality Report

We produce a final quality report summarizing the dataset. These numbers go
into the Kaggle README and the Data in Brief paper.

In [ ]:
# ============================================================
# PHASE 3 · CELL 11 — DATA QUALITY REPORT
# ============================================================

print("=" * 72)
print(f"DATASET SUMMARY — {COUNCIL_FULL}")
print("=" * 72)
print(f"Generated: {RUN_STAMP}\n")

total_rows = 0
for fname in sorted(outputs.keys()):
    df = outputs[fname]
    if isinstance(df, pd.DataFrame) and not df.empty:
        total_rows += len(df)
        print(f"  {fname:<60}  {len(df):>5} rows")
    else:
        print(f"  {fname:<60}  (empty)")

print("-" * 72)
print(f"  {'GRAND TOTAL':<60}  {total_rows:>5} rows")
print()

print("Master project table — completeness:")
print("-" * 72)
for c in projects_master.columns:
    nn  = projects_master[c].notna().sum()
    pct = 100 * nn / max(len(projects_master), 1)
    bar = "█" * int(pct / 5)
    print(f"  {c:25s}  {nn:>4}/{len(projects_master):<4}  ({pct:>5.1f}%) {bar}")

print()
print(f"Total CDF value captured: "
      f"{projects_master['approved_amount_zmw'].sum():,.2f} ZMW")
print(f"Unique sectors : {projects_master['sector'].nunique()}")
print(f"Unique wards   : {projects_master['ward'].nunique()}")
print()
print("Sector distribution:")
print(projects_master["sector"].value_counts().to_string())
print()
print("Amount band distribution:")
print(projects_master["amount_band"].value_counts().to_string())
print("=" * 72)

## Phase 3 · Section 3.11 — Post-Run Verification

We reload every CSV from disk to confirm it's readable with the pipe delimiter
and its row count matches what we intended to write. Anything that doesn't
match is flagged.

In [ ]:
# ============================================================
# PHASE 3 · CELL 12 — POST-RUN VERIFICATION
# ============================================================

print("Verifying outputs...\n")

all_ok = True
for fname, df_expected in outputs.items():
    path = OUTPUT_DIR / fname
    if not path.exists():
        if isinstance(df_expected, pd.DataFrame) and not df_expected.empty:
            print(f"  MISSING: {fname}")
            all_ok = False
        continue

    df_reloaded = pd.read_csv(path, sep="|")
    if isinstance(df_expected, pd.DataFrame) and not df_expected.empty:
        if len(df_reloaded) != len(df_expected):
            print(f"  MISMATCH: {fname} "
                  f"(disk={len(df_reloaded)}, expected={len(df_expected)})")
            all_ok = False
        else:
            print(f"  ✓ {fname:<60}  {df_reloaded.shape}")

if all_ok:
    print(f"\nAll files verified. Location: {OUTPUT_DIR.resolve()}")
else:
    print(f"\nSome files had issues.")

## Phase 3 · Summary

We've completed the full cleaning pipeline. The `outputs/` folder now contains
all files we produced, each in pipe-delimited CSV format following the
assignment's naming convention.

**Files produced:**

| File | Content |
|---|---|
| `..._idp_projects.csv` | Master strategic community project registry |
| `..._idp_strategic_areas.csv` | Four 8NDP-aligned strategic areas |
| `..._idp_goals.csv` | Sector development goals |
| `..._idp_statistics.csv` | Baseline district statistics |
| `..._idp_subprogrammes.csv` | Sub-programmes by 8NDP pillar |
| `..._idp_me_framework.csv` | M&E framework rows |
| `..._idp_cost_estimates.csv` | Multi-year cost estimates |
| `..._idp_wards.csv` | Ward profile records |
| `..._newsletter_narrative.csv` | Narrative sentences from the newsletter |
| `..._sector_by_ward.csv` | Sector × ward matrix |
| `..._timeline.csv` | Project count and funding by year |

**Next steps:**
1. Combine these outputs with teammates' CSVs into the group Kaggle dataset
2. Write the Kaggle README
3. Write the Data in Brief paper
4. Commit the notebook and CSVs to GitHub
5. Submit via Moodle

## Phase 3 · Section 3.13 — Post-Cleanup Fixes

After the main cleaning pipeline runs, a few files still have quality issues
from their extraction stages. This cell applies targeted fixes to each:

1. **`strategic_areas`** — Only 2 of 4 areas survived the junk filter because
   area names are short (2 words). We restore all 4 areas by re-reading the
   source.

2. **`cost_estimates`** — The junk filter was too strict for programmatic
   cost-table rows. We keep only the essential columns and skip the filter.

3. **`statistics`** — Some rows have tiny values (1, 2, 5) that are
   extraction noise. We keep only values >= 10.

4. **`me_framework`** — The table has 39 columns due to page-spanning
   artifacts. We keep only the columns with real content.

5. **`wards`** — The ward extraction captured a bullet point, not a ward.
   We delete this file.

6. **`citizen_priorities`** — Redundant with citizen_points. We delete this
   file.

Each fix is documented in code comments so its purpose is traceable.

In [ ]:
# ============================================================
# PHASE 3 · CELL 13 — POST-CLEANUP FIXES
# ============================================================

print("Applying post-cleanup fixes...\n")

# ---------- Fix 1: strategic_areas — restore all 4 areas ----------
print("Fix 1: strategic_areas...")

STRATEGIC_AREAS = [
    "Economic Transformation and Job Creation",
    "Human and Social Development",
    "Environmental Sustainability",
    "Good Governance Environment",
]

strategic_df = pd.DataFrame([
    {
        "strategic_area":  area,
        "page_refs":       ",".join(str(p) for p, t in idp_pages
                                     if area.lower() in t.lower()),
        "source_doc":      "IDP_MAIN",
    }
    for area in STRATEGIC_AREAS
])
strategic_df.to_csv(CLEAN_DIR / "strategic_areas_clean.csv",
                    sep="|", index=False)
print(f"  ✅ strategic_areas: {len(strategic_df)} rows")


# ---------- Fix 2: cost_estimates — keep essential columns ----------
print("\nFix 2: cost_estimates...")
ce = extracted.get("cost_estimates", pd.DataFrame()).copy()
if not ce.empty:
    keep = [c for c in ["programme", "year", "cost_zmw", "page", "source_doc"]
            if c in ce.columns]
    ce = ce[keep]
    if "year" in ce.columns:
        ce["year"] = pd.to_numeric(ce["year"], errors="coerce").astype("Int64")
    if "cost_zmw" in ce.columns:
        ce["cost_zmw"] = pd.to_numeric(ce["cost_zmw"], errors="coerce")
    if "programme" in ce.columns:
        ce = ce[ce["programme"].notna()
                & (ce["programme"].astype(str).str.len() >= 3)]
    if "cost_zmw" in ce.columns:
        ce = ce[ce["cost_zmw"].notna() & (ce["cost_zmw"] > 0)]
    ce.to_csv(CLEAN_DIR / "cost_estimates_clean.csv", sep="|", index=False)
    print(f"  ✅ cost_estimates: {len(ce)} rows")
else:
    print("  ⚠️  cost_estimates: empty in extracted")


# ---------- Fix 3: statistics — drop tiny values ----------
print("\nFix 3: statistics...")
st = extracted.get("statistics", pd.DataFrame()).copy()
if not st.empty:
    if "value" in st.columns:
        st["value"] = pd.to_numeric(st["value"], errors="coerce")
        st = st[st["value"].notna() & (st["value"] >= 10)]
    if "context" in st.columns:
        st["context"] = st["context"].apply(
            lambda v: clean_text(v, max_len=200) if isinstance(v, str) else v
        )
    st.to_csv(CLEAN_DIR / "statistics_clean.csv", sep="|", index=False)
    print(f"  ✅ statistics: {len(st)} rows")
else:
    print("  ⚠️  statistics: empty in extracted")


# ---------- Fix 4 (v2): me_framework — WHITELIST columns ----------
print("\nFix 4: me_framework...")
me = extracted.get("me_framework", pd.DataFrame()).copy()
if not me.empty:
    # Whitelist: keep ONLY columns with these exact names
    USEFUL = ["page", "source_doc", "Strategies", "Program", "Activities",
              "Location", "Baseline", "Target", "Responsible",
              "Indicator", "Input", "Freq"]
    me = me[[c for c in me.columns if c in USEFUL]]

    # Drop columns that are entirely empty
    empty = [c for c in me.columns
             if me[c].apply(lambda v: pd.isna(v) or str(v).strip() == "").all()]
    me = me.drop(columns=empty, errors="ignore")

    # Keep only rows where at least one content column has data
    content_cols = [c for c in ["Activities", "Program", "Indicator",
                                 "Strategies"] if c in me.columns]
    if content_cols:
        mask = me[content_cols].notna().any(axis=1)
        me = me[mask].reset_index(drop=True)

    me.to_csv(CLEAN_DIR / "me_framework_clean.csv", sep="|", index=False)
    print(f"  ✅ me_framework: {len(me)} rows, {me.shape[1]} cols")
    print(f"     Columns kept: {list(me.columns)}")
else:
    print("  ⚠️  me_framework: empty in extracted")


# ---------- Fix 5: delete wards (junk) ----------
print("\nFix 5: wards...")
wards_path = CLEAN_DIR / "wards_clean.csv"
if wards_path.exists():
    wards_path.unlink()
    print(f"  🗑️  Deleted {wards_path.name} (junk content)")
else:
    print("  ✓  wards_clean.csv already absent")


# ---------- Fix 6: delete citizen_priorities (redundant) ----------
print("\nFix 6: citizen_priorities...")
for p in [CLEAN_DIR / "citizen_priorities_clean.csv"]:
    if p.exists():
        p.unlink()
        print(f"  🗑️  Deleted {p.name} (redundant with citizen_points)")
    else:
        print(f"  ✓  {p.name} already absent")


# ---------- Fix 7: citizen_points — deduplicate and drop fragments ----------
print("\nFix 7: citizen_points...")
cp = extracted.get("citizen_points", pd.DataFrame()).copy()
if not cp.empty and "point" in cp.columns:
    # Clean text
    cp["point"] = cp["point"].apply(clean_text)

    # Drop fragments: length < 20 chars
    cp = cp[cp["point"].notna()
            & (cp["point"].astype(str).str.len() >= 20)]

    # Remove TOC-style fragments
    cp = cp[~cp["point"].astype(str).str.contains(r"\.{3,}", na=False, regex=True)]

    # Deduplicate
    cp = cp.drop_duplicates(subset=["point"]).reset_index(drop=True)

    # Save
    cp.to_csv(CLEAN_DIR / "citizen_points_clean.csv", sep="|", index=False)
    print(f"  ✅ citizen_points: {len(cp)} rows")
else:
    print("  ⚠️  citizen_points: empty in extracted")


print("\n✅ Post-cleanup fixes applied.")

## Phase 3 · Section 3.13b — Targeted Fixes for me_framework & citizen_points

Two files still carry too much noise after the main cleanup:

**`me_framework_clean.csv`** — 21 rows × 12 columns. Most columns are
sparse (fewer than 30% of rows have data). We drop any column that is
populated in less than 30% of rows, keeping only `page`, `source_doc`
plus the columns that actually carry content.

**`citizen_points_clean.csv`** — 77 rows. Some are real community
priority statements; some are fragments of the introduction. We apply
a second-pass filter that keeps only rows that:
  - Start with a capital letter
  - Contain 6+ words
  - Do NOT start with a banned prefix (`Due to`, `Poor`, `Not enough`,
    `Majority`, `Reduction in`, `Attracts`, `Land telephone`, etc.)
  - Do NOT end with a colon or a dangling preposition

In [ ]:
# ============================================================
# PHASE 3 · CELL 13b — TARGETED FIXES (v2 with bullet splitting)
# ============================================================

print("Applying targeted fixes...\n")


# ============================================================
# FIX A: me_framework — drop sparse columns
# ============================================================
print("Fix A: me_framework — drop sparse columns...")

me = extracted.get("me_framework", pd.DataFrame()).copy()
if me.empty:
    print("  ⚠️  me_framework is empty in extracted")
else:
    # Whitelist: keep only known-useful column names
    USEFUL = ["page", "source_doc", "Strategies", "Program", "Activities",
              "Location", "Baseline", "Target", "Responsible",
              "Indicator", "Input", "Freq"]
    me = me[[c for c in me.columns if c in USEFUL]]

    # These columns are always kept, even if sparse
    MUST_KEEP = {"page", "source_doc"}

    # Drop columns where fewer than 30% of rows have content
    n_rows = len(me)
    min_filled = n_rows * 0.30

    sparse_cols = []
    for c in me.columns:
        if c in MUST_KEEP:
            continue
        filled = me[c].apply(
            lambda v: not (pd.isna(v) or str(v).strip() == "")
        ).sum()
        if filled < min_filled:
            sparse_cols.append(c)

    me = me.drop(columns=sparse_cols, errors="ignore")
    print(f"  Dropped sparse columns: {sparse_cols}")

    # Drop rows where every content column is empty
    content_cols = [c for c in me.columns if c not in MUST_KEEP]
    if content_cols:
        mask = me[content_cols].apply(
            lambda row: any(
                not (pd.isna(v) or str(v).strip() == "") for v in row
            ),
            axis=1,
        )
        me = me[mask].reset_index(drop=True)

    me.to_csv(CLEAN_DIR / "me_framework_clean.csv", sep="|", index=False)
    print(f"  ✅ me_framework: {len(me)} rows, {me.shape[1]} cols")
    print(f"     Columns kept: {list(me.columns)}")


# ============================================================
# FIX B: citizen_points — strict filter + bullet splitting
# ============================================================
print("\nFix B: citizen_points — strict filter with bullet splitting...")

cp = extracted.get("citizen_points", pd.DataFrame()).copy()

if cp.empty or "point" not in cp.columns:
    print("  ⚠️  citizen_points is empty or missing 'point' column")
else:
    # ---------- 1. Clean text ----------
    cp["point"] = cp["point"].apply(clean_text)

    # ---------- 2. Drop nulls and short fragments ----------
    cp = cp[
        cp["point"].notna()
        & (cp["point"].astype(str).str.len() >= 25)
    ]

    # ---------- 3. KEY STEP: split on bullet and keep the action ----------
    # Many Citizen IDP rows look like:
    #   "Absence of a farmer training center. • Construct one Farmer training center"
    #   "High dependence on rains. • Construct a dam in Munyama Block by 2025."
    # We keep only the part AFTER the bullet — the proposed action.
    def keep_action_after_bullet(text):
        if not isinstance(text, str):
            return text
        if "•" in text:
            parts = [p.strip() for p in text.split("•") if p.strip()]
            if parts:
                # Return the longest part (usually the action)
                return max(parts, key=len)
        return text

    cp["point"] = cp["point"].apply(keep_action_after_bullet)

    # Re-clean after split
    cp["point"] = cp["point"].apply(clean_text)

    # Drop again if the action part is too short
    cp = cp[
        cp["point"].notna()
        & (cp["point"].astype(str).str.len() >= 20)
    ]

    # ---------- 4. Must start with a capital letter ----------
    cp = cp[cp["point"].astype(str).str[0].str.isupper()]

    # ---------- 5. Must have 5+ words (relaxed to 5) ----------
    cp = cp[cp["point"].astype(str).str.split().str.len() >= 5]

    # ---------- 6. Must NOT end with colon or dangling preposition ----------
    bad_endings = r"(:\s*$|\b(for|of|in|to|with|and|by|at|on|the|a|an|from|as)\s*$)"
    cp = cp[~cp["point"].astype(str).str.contains(
        bad_endings, regex=True, case=False, na=False
    )]

    # ---------- 7. Must NOT contain TOC leaders ----------
    cp = cp[~cp["point"].astype(str).str.contains(
        r"\.{3,}", regex=True, na=False
    )]

    # ---------- 8. Must NOT start with these noise prefixes ----------
    BAD_PREFIXES = [
        "due to", "poor ", "not enough", "majority", "reduction in",
        "attracts ", "land telephone", "lack of", "increased use",
        "onsite sanitation", "over ", "to create", "the extension",
        "the district", "the council", "in 2023", "in 2024", "in 2025",
    ]
    lower = cp["point"].astype(str).str.lower()
    for prefix in BAD_PREFIXES:
        cp = cp[~lower.str.startswith(prefix)]
        # Recompute lower after each filter to stay in sync with cp
        lower = cp["point"].astype(str).str.lower()

    # ---------- 9. Must NOT be an ALL-CAPS heading ----------
    cp = cp[~cp["point"].astype(str).str.match(
        r"^[A-Z\s\.\-]+$", na=False
    )]

    # ---------- 10. Deduplicate ----------
    cp = cp.drop_duplicates(subset=["point"]).reset_index(drop=True)

    # ---------- 11. Save ----------
    cp.to_csv(CLEAN_DIR / "citizen_points_clean.csv", sep="|", index=False)
    print(f"  ✅ citizen_points: {len(cp)} rows")
    print(f"     Sample (first 10):")
    for s in cp["point"].head(10):
        print(f"       • {s[:90]}")


print("\n✅ Targeted fixes applied.")

In [ ]:
# ============================================================
# VERIFY CDF EXTRACTION WITH 'LINES' STRATEGY
# ============================================================

from pathlib import Path
import pdfplumber
import pandas as pd

# Resolve paths
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_DIR = REPO_ROOT / "data" / "raw" / "idp"
CDF_PATH = RAW_DIR / "2025-Approved-Community-Projects_Kabwe-Central.pdf"

# Use the same extract_all_tables from Phase 2 · Cell 2
cdf_tables = extract_all_tables(CDF_PATH)

print(f"Tables detected: {len(cdf_tables)}\n")
for i, t in enumerate(cdf_tables):
    print(f"Table {i+1}: shape={t.shape}")
    print(f"  Columns: {list(t.columns)}")
    print(f"  First data row: {list(t.iloc[0].values)[:6]}")
    print()

# Show the main project table
if cdf_tables:
    main = cdf_tables[0]
    print(f"\n{'=' * 70}")
    print(f"MAIN PROJECT TABLE — first 5 rows")
    print(f"{'=' * 70}")
    print(main.head().to_string())

In [ ]:
from pathlib import Path
import pandas as pd

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

# Check extracted files
print("data/extracted/:")
for f in sorted((REPO_ROOT / "data" / "extracted").glob("*.csv")):
    df = pd.read_csv(f, sep="|")
    print(f"  {f.name:<40}  {df.shape}")

print("\noutputs/:")
for f in sorted((REPO_ROOT / "outputs").glob("*.csv")):
    df = pd.read_csv(f, sep="|")
    print(f"  {f.name:<55}  {df.shape}")

In [ ]:
import pdfplumber
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
CDF_PATH = REPO_ROOT / "data" / "raw" / "idp" / "2025-Approved-Community-Projects_Kabwe-Central.pdf"

LINE_SETTINGS = {"vertical_strategy": "lines", "horizontal_strategy": "lines"}

with pdfplumber.open(CDF_PATH) as pdf:
    for pno, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables(LINE_SETTINGS)
        print(f"\n{'=' * 70}")
        print(f"PAGE {pno}: {len(tables)} tables")
        print(f"{'=' * 70}")
        for i, t in enumerate(tables):
            if not t:
                continue
            print(f"\n  Table {i+1}: {len(t)} rows × {len(t[0])} cols")
            print(f"    Header: {[str(c)[:25] for c in t[0] if c]}")
            # Show first 2 data rows
            for j, row in enumerate(t[1:3], start=1):
                print(f"    Row {j}: {[str(c)[:25] for c in row if c][:4]}")

In [ ]:
import pandas as pd
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
CLEAN_DIR = REPO_ROOT / "data" / "clean"

print("data/clean/ contents:")
for f in sorted(CLEAN_DIR.glob("*.csv")):
    df = pd.read_csv(f, sep="|")
    print(f"  {f.name:<40} {df.shape}")

In [ ]:
import pandas as pd
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "outputs"

print("=" * 72)
print("KABWE MUNICIPAL COUNCIL — FINAL DATASET")
print("=" * 72)

total = 0
files = sorted(OUTPUT_DIR.glob("*.csv"))
for f in files:
    df = pd.read_csv(f, sep="|")
    total += len(df)
    print(f"  {f.name:<58} {len(df):>5} rows × {df.shape[1]:>2} cols")

print("-" * 72)
print(f"  {'TOTAL':<58} {total:>5} rows across {len(files)} files")
print("=" * 72)

# Sanity checks
print("\nSanity checks:")

# 1. Total CDF value
proj = pd.read_csv(OUTPUT_DIR / "db-unza26-csc4792-kabwe_idp_projects.csv", sep="|")
total_value = proj["approved_amount_zmw"].sum()
print(f"  ✓ Total funding captured:  {total_value:,.2f} ZMW")

# 2. Sector distribution
print(f"\n  Sector distribution in projects.csv:")
for sector, count in proj["sector"].value_counts().items():
    print(f"    {sector:<25} {count}")

# 3. No empty files
empty_files = [f.name for f in files if pd.read_csv(f, sep="|").empty]
if empty_files:
    print(f"\n  ⚠️  Empty files: {empty_files}")
else:
    print(f"\n  ✓ No empty files")

In [ ]:
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "outputs"

# Files that should NOT exist in the final output
STALE = [
    "db-unza26-csc4792-kabwe_idp_wards.csv",
    "db-unza26-csc4792-kabwe_idp_citizen_priorities.csv",
    "db-unza26-csc4792-kabwe_idp_budget.csv",
]

for fname in STALE:
    p = OUTPUT_DIR / fname
    if p.exists():
        p.unlink()
        print(f"🗑️  Deleted {fname}")
    else:
        print(f"✓  {fname} not present")

In [ ]:
import pandas as pd
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "outputs"

print("=" * 72)
print("KABWE MUNICIPAL COUNCIL — FINAL DATASET")
print("=" * 72)

total = 0
files = sorted(OUTPUT_DIR.glob("*.csv"))
for f in files:
    df = pd.read_csv(f, sep="|")
    total += len(df)
    print(f"  {f.name:<58} {len(df):>5} rows × {df.shape[1]:>2} cols")

print("-" * 72)
print(f"  {'TOTAL':<58} {total:>5} rows across {len(files)} files")
print("=" * 72)